# Exploration: data checks

In [2]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.utils import load_config, load_data

cfg = load_config()
Col = cfg["columns"]
train = load_data(cfg["paths"]["train"], cfg)
val = load_data(cfg["paths"]["validation"], cfg)

## Shape, dates and column types

In [ ]:
# Prints the dimensions of rows and columns, the date range, and the columns' data types.
def check_shape_and_dates(df, name):
    print(f"[{name}] shape: {df.shape}")
    print(f"[{name}] dates: {df[Col['Date']].min().date()} -> {df[Col['Date']].max().date()}")
    print(df.dtypes.to_string())

check_shape_and_dates(train, "train")
check_shape_and_dates(val, "validation")

[train] shape: (48000, 14)
[train] dates: 2025-01-01 -> 2025-10-31
load_id                 object
pickup                  object
delivery                object
pickup_lat             float64
pickup_lon             float64
delivery_lat           float64
delivery_lon           float64
distance               float64
equipment               object
weight                 float64
date            datetime64[ns]
market_index           float64
quote_signal           float64
posted_rate            float64
[validation] shape: (12000, 13)
[validation] dates: 2025-11-01 -> 2025-12-31
load_id                 object
pickup                  object
delivery                object
pickup_lat             float64
pickup_lon             float64
delivery_lat           float64
delivery_lon           float64
distance               float64
equipment               object
weight                 float64
date            datetime64[ns]
market_index           float64
quote_signal           float64


## Duplicates

In [4]:
# Prints how many load ids and how many full rows are duplicated.
def check_duplicates(df, name):
    print(f"[{name}] duplicate ids : {df[Col['Load_Id']].duplicated().sum()}")
    print(f"[{name}] duplicate rows: {df.duplicated().sum()}")

check_duplicates(train, "train")
check_duplicates(val, "validation")

[train] duplicate ids : 0
[train] duplicate rows: 0
[validation] duplicate ids : 0
[validation] duplicate rows: 0


## Missing values

In [5]:
# Prints the count of missing values per column and the load ids of the rows where they are missing.
def check_missing(df, name):
    print(f"[{name}] missing values")
    for col in df.columns:
        ids = df.loc[df[col].isna(), Col["Load_Id"]].tolist()
        if ids:
            print(f"{col}: {len(ids)} missing, load_ids: {ids}")

check_missing(train, "train")
check_missing(val, "validation")

[train] missing values
weight: 300 missing, load_ids: ['TR-000324', 'TR-000649', 'TR-000695', 'TR-000733', 'TR-000912', 'TR-000943', 'TR-000964', 'TR-001010', 'TR-001022', 'TR-001059', 'TR-001133', 'TR-001140', 'TR-001208', 'TR-001254', 'TR-001737', 'TR-001857', 'TR-001909', 'TR-001924', 'TR-001932', 'TR-001965', 'TR-001996', 'TR-002065', 'TR-002509', 'TR-002905', 'TR-003017', 'TR-003051', 'TR-003531', 'TR-003599', 'TR-003717', 'TR-003915', 'TR-004288', 'TR-004322', 'TR-004475', 'TR-004722', 'TR-004961', 'TR-004984', 'TR-005016', 'TR-005065', 'TR-005585', 'TR-005604', 'TR-005672', 'TR-005859', 'TR-005945', 'TR-006029', 'TR-006408', 'TR-006563', 'TR-006729', 'TR-006906', 'TR-006943', 'TR-007076', 'TR-007285', 'TR-007322', 'TR-007337', 'TR-007373', 'TR-007433', 'TR-007828', 'TR-007982', 'TR-008327', 'TR-008390', 'TR-008395', 'TR-008413', 'TR-008439', 'TR-008583', 'TR-008666', 'TR-008700', 'TR-008963', 'TR-009215', 'TR-009457', 'TR-010143', 'TR-010239', 'TR-010355', 'TR-010482', 'TR-01051

## Negative values

In [13]:
# Prints the count of negative values per target column and the load ids where they occur.
def check_negatives(df, name):
    print(f"[{name}] negative values")
    cols_to_check = [Col["Weight"], Col["Distance"], Col.get("Posted_Rate", "posted_rate")]
    for col in cols_to_check:
        if col in df.columns:
            ids = df.loc[df[col] < 0, Col["Load_Id"]].tolist()
            if ids:
                print(f"{col}: {len(ids)} negative, load_ids: {ids}")
            else:
                print(f"{col}: 0 negative")

check_negatives(train, "train")
check_negatives(val, "validation")

[train] negative values
weight: 292 negative, load_ids: ['TR-000069', 'TR-000205', 'TR-000207', 'TR-000226', 'TR-000377', 'TR-000447', 'TR-000496', 'TR-000642', 'TR-000807', 'TR-001094', 'TR-001348', 'TR-001351', 'TR-001530', 'TR-001619', 'TR-001688', 'TR-001717', 'TR-001836', 'TR-001897', 'TR-001929', 'TR-001940', 'TR-002263', 'TR-002288', 'TR-003149', 'TR-003208', 'TR-003345', 'TR-003464', 'TR-003585', 'TR-003626', 'TR-004369', 'TR-004701', 'TR-004779', 'TR-004941', 'TR-005035', 'TR-005116', 'TR-005153', 'TR-005237', 'TR-005665', 'TR-005764', 'TR-006081', 'TR-006231', 'TR-006354', 'TR-006356', 'TR-006485', 'TR-006530', 'TR-006982', 'TR-007001', 'TR-007121', 'TR-007124', 'TR-007176', 'TR-007407', 'TR-007988', 'TR-008072', 'TR-008171', 'TR-008223', 'TR-008385', 'TR-008669', 'TR-008680', 'TR-008746', 'TR-008833', 'TR-008922', 'TR-009115', 'TR-009295', 'TR-009373', 'TR-009791', 'TR-009815', 'TR-009943', 'TR-010235', 'TR-010371', 'TR-010376', 'TR-010397', 'TR-010481', 'TR-010484', 'TR-010

## Coordinates

In [7]:
# Prints whether each city has one fixed coordinate.
def check_coordinates(df, name):
    groups = [
                (Col["Pickup"], Col["Pickup_Lat"], Col["Pickup_Lon"]),
                (Col["Delivery"], Col["Delivery_Lat"], Col["Delivery_Lon"]),
             ]
    for city, lat, lon in groups:
        per_city = df.groupby(city)[[lat, lon]].nunique()
        not_fixed = (per_city > 1).any(axis=1).sum()
        print(f"[{name}] {city}: cities with more than 1 coordinate value: {not_fixed}")

check_coordinates(train, "train")
check_coordinates(val, "validation")

[train] pickup: cities with more than 1 coordinate value: 0
[train] delivery: cities with more than 1 coordinate value: 0
[validation] pickup: cities with more than 1 coordinate value: 0
[validation] delivery: cities with more than 1 coordinate value: 0


## Outliers (IQR rule)

In [8]:
# Prints the IQR fences and how many values fall below or above them for weight, distance, and posted rate.
def check_outliers(df, name):
    d = df.copy()
    d[Col["Weight"]] = d[Col["Weight"]].abs()
    print(f"[{name}] outliers (IQR rule: 1.5 x IQR beyond Q1 and Q3)")
    cols_to_check = [Col["Distance"], Col["Weight"], Col.get("Posted_Rate", "posted_rate")]
    for col in cols_to_check:
        if col in d.columns:
            s = d[col].dropna()
            q1, q3 = s.quantile([0.25, 0.75])
            iqr = q3 - q1
            low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
            print(f"{col}: range {s.min():.3f} -> {s.max():.3f} | fences {low:.3f} -> {high:.3f} | below {(s < low).sum()}, above {(s > high).sum()}")

check_outliers(train, "train")
check_outliers(val, "validation")

[train] outliers (IQR rule: 1.5 x IQR beyond Q1 and Q3)
distance: range 70.000 -> 3439.800 | fences -1092.287 -> 3288.213 | below 0, above 32
weight: range 5000.000 -> 47500.000 | fences 9210.125 -> 53775.125 | below 149, above 0
posted_rate: range 57.220 -> 25533.000 | fences -1867.238 -> 6449.543 | below 0, above 260
[validation] outliers (IQR rule: 1.5 x IQR beyond Q1 and Q3)
distance: range 70.000 -> 3325.700 | fences -1115.500 -> 3342.100 | below 0, above 0
weight: range 5000.000 -> 47500.000 | fences 8975.250 -> 53717.250 | below 33, above 0


## Correlation

In [12]:
# Prints the correlation between the numeric inputs.
def check_correlation(df):
    d = df.copy()
    d[Col["Weight"]] = d[Col["Weight"]].abs()
    d["log_distance"] = np.log(d[Col["Distance"]])
    d["log_rate"] = np.log(d[Col["Target"]])
    cols = [
            "log_distance", Col["Weight"], Col["Market_Index"], Col["Quote_Signal"],
            Col["Pickup_Lat"], Col["Pickup_Lon"], Col["Delivery_Lat"], Col["Delivery_Lon"], "log_rate",
           ]
    print("[train] correlation (raw, all rows)")
    print(d[cols].corr().round(3).to_string())

check_correlation(train)

[train] correlation (raw, all rows)
              log_distance  weight  market_index  quote_signal  pickup_lat  pickup_lon  delivery_lat  delivery_lon  log_rate
log_distance         1.000   0.004        -0.002        -0.065      -0.076      -0.237        -0.079        -0.238     0.967
weight               0.004   1.000         0.003         0.006      -0.000       0.004        -0.004         0.001     0.042
market_index        -0.002   0.003         1.000        -0.068      -0.002       0.001        -0.003        -0.008     0.031
quote_signal        -0.065   0.006        -0.068         1.000      -0.004       0.014        -0.003         0.011    -0.052
pickup_lat          -0.076  -0.000        -0.002        -0.004       1.000       0.559        -0.008        -0.006    -0.097
pickup_lon          -0.237   0.004         0.001         0.014       0.559       1.000        -0.009        -0.012    -0.241
delivery_lat        -0.079  -0.004        -0.003        -0.003      -0.008      -0.009   